# 03 - Silver Layer: Customers, Stores, Transactions

**Project:** Retail Analytics & Product Dimension History

## What this notebook does
Standard cleanup for the three straightforward tables. The Products
dimension gets its own dedicated SCD Type 2 treatment in a separate
notebook (04_retail_scd2_products), since that logic deserves focused
attention rather than being mixed in here.

## Tables created
- `main.retail_analytics.silver_customers`
- `main.retail_analytics.silver_stores`
- `main.retail_analytics.silver_transactions`

In [0]:
from pyspark.sql.functions import col, datediff, current_date

bronze_customers = spark.table("main.retail_analytics.bronze_customers")

silver_customers = (
    bronze_customers
    .withColumn("age", (datediff(current_date(), col("BirthDate")) / 365.25).cast("int"))
    .withColumn("tenure_days", datediff(current_date(), col("JoinDate")))
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("FirstName", "first_name")
    .withColumnRenamed("LastName", "last_name")
    .withColumnRenamed("Gender", "gender")
    .withColumnRenamed("BirthDate", "birth_date")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("JoinDate", "join_date")
)

(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.retail_analytics.silver_customers")
)
print("Row count:", spark.table("main.retail_analytics.silver_customers").count())

Row count: 200


In [0]:
bronze_stores = spark.table("main.retail_analytics.bronze_stores")

silver_stores = (
    bronze_stores
    .withColumnRenamed("StoreID", "store_id")
    .withColumnRenamed("StoreName", "store_name")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("Region", "region")
)

(
    silver_stores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.retail_analytics.silver_stores")
)
print("Row count:", spark.table("main.retail_analytics.silver_stores").count())

Row count: 5


In [0]:
from pyspark.sql.functions import round as _round

bronze_transactions = spark.table("main.retail_analytics.bronze_transactions")

silver_transactions = (
    bronze_transactions
    .withColumnRenamed("TransactionID", "transaction_id")
    .withColumnRenamed("Date", "transaction_date")
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("ProductID", "product_id")
    .withColumnRenamed("StoreID", "store_id")
    .withColumnRenamed("Quantity", "quantity")
    .withColumnRenamed("Discount", "discount_pct")
    .withColumnRenamed("PaymentMethod", "payment_method")
)

(
    silver_transactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.retail_analytics.silver_transactions")
)
print("Row count:", spark.table("main.retail_analytics.silver_transactions").count())

Row count: 5000
